In [2]:
import requests
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import cv2
import numpy as np
import pandas as pd
from skimage import feature
from skimage.util import img_as_ubyte 
from skimage.feature import local_binary_pattern
from scipy.stats import entropy
from tqdm import tqdm
import re
import random
from selenium.webdriver.chrome.options import Options

In [3]:
url_dict ={
    "emmy_kasbit": ["https://www.bellanaija.com/2025/11/lagos-fashion-week-2025-see-emmy-kasbits-collection/"],
    "fruche": ["https://www.bellanaija.com/2025/11/lagos-fashion-week-2025-see-fruches-collection/"],
    "imad_eduso": ["https://www.bellanaija.com/2025/11/lagos-fashion-week-2025-see-imad-edusos-collection/"],
    "hertunba": ["https://www.bellanaija.com/2025/11/lagos-fashion-week-2025-see-hertunbas-collection/"],
    "y'wande": ["https://www.bellanaija.com/2025/11/lagos-fashion-week-2025-see-ywandes-collection/"],
    "wote": ["https://www.bellanaija.com/2025/11/lagos-fashion-week-2025-see-wotes-collection/"],
    "ajanee": ["https://www.bellanaija.com/2025/11/lagos-fashion-week-2025-see-ajanees-collection/"],
    "lila_bare": ["https://www.bellanaija.com/2025/11/lagos-fashion-week-2025-see-lila-bares-collection/"],
    "the_or_foundation": ["https://www.bellanaijastyle.com/the-or-foundation-lagos-fw-2025/"],
    "rendoll": ["https://www.bellanaija.com/2025/11/lagos-fashion-week-2025-see-rendolls-collection/"],
    "boyedoe": ["https://www.bellanaija.com/2025/11/lagos-fashion-week-2025-see-boyedoes-collection/"],
    "nya": ["https://www.bellanaija.com/2025/11/lagos-fashion-week-2025-see-nyas-collection/"],
    "dimeji_ilori": ["https://www.bellanaija.com/2025/11/lagos-fashion-week-2025-see-dimeji-iloris-collection/"],
    "revival_london": ["https://www.bellanaija.com/2025/11/lagos-fashion-week-2025-see-revivals-collection/"],
    "lb_lumina": ["https://www.bellanaija.com/2025/11/lagos-fashion-week-2025-see-luminas-collection/"],
    "studio_imo": ["https://www.bellanaijastyle.com/studio-imo-lagos-fw-2025/"],
    "hawa_paris": ["https://www.bellanaijastyle.com/hawa-lagos-fw-2025/"],
    "bloke": ["https://lagosfashionweek.ng/seasons/lagosfw-2025/bloke"],
    "jzo":["https://www.bellanaijastyle.com/lagos-fashion-week-2025-jzo/"],
    "cute_saint": ["https://lagosfashionweek.ng/seasons/lagosfw-2025/cute-saint"],
    "olooh": ["https://www.bellanaijastyle.com/lagos-fashion-week-2025-olooh/"],
    "lfj": ["https://www.bellanaijastyle.com/1247410-2/"],
    "sevon_dejana": ["https://www.bellanaijastyle.com/lagos-fashion-week-2025-sevon-dejana/"],
    "ibilola_ogundipe": ["https://www.bellanaijastyle.com/ibilola-ogundipe-lagos-fashion-week-2025-collection/"],
    "pepperrow": ["https://www.bellanaijastyle.com/lagos-fashion-week-2025-pepper-row/"],
    "cynthia_abila": ["https://www.bellanaijastyle.com/cynthia-abila-lagos-fw-2025/"],
    "eki_silk": ["https://www.bellanaijastyle.com/eki-silk-lagos-fw-2025/"],
    "adama_paris": ["https://www.bellanaijastyle.com/adama-paris-lagos-fw-2025/"],
    "desiree_iyama": ["https://www.bellanaijastyle.com/desiree-iyama-lagos-fw-2025/"],
    "maxjenny": ["https://www.bellanaijastyle.com/maxjenny-lagos-fw-2025/"],
    "babayo": ["https://www.bellanaijastyle.com/babayo-lagos-fw-2025/"],
    "eso_by_liman": ["https://www.bellanaijastyle.com/eso-liman-lagos-fw-2025/"],
    "oshobor": ["https://www.bellanaijastyle.com/oshobor-lagos-fw-2025/"],
    "pettre_taylor": ["https://www.bellanaijastyle.com/lagos-fashion-week-2025-pettre-taylor/"],
    "elexiay": ["https://www.bellanaijastyle.com/elexiay-lagos-fw-2025/"],
    "sahrazad": ["https://www.bellanaijastyle.com/sahrazad-lagos-fashion-week-2025/"],
    "maison_alulla": ["https://www.bellanaijastyle.com/maison-allula-lagos-fw-2025/"],
    "nkwo": ["https://www.bellanaijastyle.com/nkwo-lagos-fw-2025/"],
    "ajabeng": ["https://www.bellanaijastyle.com/ajabeng-lagos-fw-2025/"],
    "mot_the_label": ["https://www.bellanaijastyle.com/mot-lagos-fw-2025/"],
    "for_style_sake": ["https://www.bellanaijastyle.com/fss-for-style-sake-lagos-fw-2025/"],
    "street_souk":["https://www.bellanaijastyle.com/1247408-2/"],
    "last_three": ["https://soul957fm.com/2025/11/07/lagos-fashion-week-2025-green-access/#ndiiche-x-sinae"],
    }

In [6]:
# --- CONFIGURATION ---
data_path = r'C:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\data\runway'
os.makedirs(data_path, exist_ok=True)

# Set up Chrome options to be less "bot-like"
chrome_options = Options()
chrome_options.add_argument("--window-size=1920,1080")
# chrome_options.add_argument("--headless") # Uncomment to run without a window opening

driver = webdriver.Chrome(options=chrome_options)
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

# --- MAIN LOOP ---
for brand_name, urls in url_dict.items():
    print(f"\nProcessing brand: {brand_name}")
    
    folder = os.path.join(data_path, brand_name)
    if os.path.exists(folder) and len(os.listdir(folder)) > 0:
        print(f"Folder for {brand_name} already exists and is not empty. Skipping.")
        continue
    
    os.makedirs(folder, exist_ok=True)
    download_count = 0

    for url in urls:
        try:
            driver.get(url)
            # Smooth scrolling to trigger lazy-loaded images
            total_height = driver.execute_script("return document.body.scrollHeight")
            for i in range(1, total_height, 800):
                driver.execute_script(f"window.scrollTo(0, {i});")
                time.sleep(0.3)
            time.sleep(2) # Final wait for all assets

            # Broader selector to catch standard posts and special sponsor pages
            elements = driver.find_elements(By.CSS_SELECTOR, "article img, .entry-content img, .gallery-item img, .post-content img")
            print(f"Found {len(elements)} images for {brand_name}.")

            for index, img in enumerate(elements):
                # Check all possible sources (Lazy loading often hides the link in data-src)
                src = (img.get_attribute('data-lazy-src') or 
                       img.get_attribute('data-src') or 
                       img.get_attribute('srcset') or 
                       img.get_attribute('src'))

                if src and "http" in src:
                    # 1. If it's a srcset, pick the first URL
                    if "," in src:
                        src = src.split(',')[0].split(' ')[0]

                    # 2. Filter: Only keep images from the WordPress uploads folder
                    # This ignores icons, tracking pixels, and sidebars
                    if "wp-content/uploads" not in src or "logo" in src.lower():
                        continue

                    # 3. UPGRADE: Strip thumbnail dimensions to get the original high-res photo
                    # e.g., 'image-300x300.jpg' -> 'image.jpg'
                    high_res_url = re.sub(r'-\d+x\d+(?=\.(jpg|jpeg|png|webp))', '', src)

                    try:
                        response = requests.get(high_res_url, headers=headers, timeout=15)
                        if response.status_code == 200:
                            # Use a unique filename in case of multiple URLs per brand
                            filename = f"{brand_name}_{download_count + 1}.jpg"
                            path = os.path.join(folder, filename)
                            
                            with open(path, 'wb') as f:
                                f.write(response.content)
                            download_count += 1
                    except Exception as e:
                        print(f"Error downloading {high_res_url}: {e}")

        except Exception as e:
            print(f"Error accessing {url}: {e}")

    print(f" Finished {brand_name}: Downloaded {download_count} images.")
    # Random sleep to avoid being flagged for aggressive scraping
    time.sleep(random.uniform(1, 3))

driver.quit()
print("\n✨ All brands processed!")


Processing brand: emmy_kasbit
Found 54 images for emmy_kasbit.
 Finished emmy_kasbit: Downloaded 51 images.

Processing brand: fruche
Found 46 images for fruche.
 Finished fruche: Downloaded 43 images.

Processing brand: imad_eduso
Found 45 images for imad_eduso.
 Finished imad_eduso: Downloaded 42 images.

Processing brand: hertunba
Found 55 images for hertunba.
 Finished hertunba: Downloaded 52 images.

Processing brand: y'wande
Found 46 images for y'wande.
 Finished y'wande: Downloaded 43 images.

Processing brand: wote
Found 46 images for wote.
 Finished wote: Downloaded 43 images.

Processing brand: ajanee
Found 32 images for ajanee.
 Finished ajanee: Downloaded 29 images.

Processing brand: lila_bare
Found 50 images for lila_bare.
 Finished lila_bare: Downloaded 47 images.

Processing brand: the_or_foundation
Found 37 images for the_or_foundation.
 Finished the_or_foundation: Downloaded 35 images.

Processing brand: rendoll
Found 31 images for rendoll.
 Finished rendoll: Downloa

FABRIC EXTRACTION

GLCM EXTRACTION

In [2]:
def extract_rgb_from_image(image):
    B, G, R = cv2.split(image)
    mean_R = np.mean(R)
    mean_G = np.mean(G)
    mean_B = np.mean(B)
    return mean_R, mean_G, mean_B

In [3]:
def calculate_glcm_features(image_channel):
    bins = np.array([0, 16, 32, 48, 64, 80, 96, 112, 128,
                     144, 160, 176, 192, 208, 224, 240, 255])
    inds = np.digitize(image_channel, bins)
    max_value = inds.max() + 1
    glcm = feature.graycomatrix(inds, [1], [0, np.pi/4, np.pi/2, 3*np.pi/4],
                                levels=max_value, normed=True, symmetric=True)

    contrast = feature.graycoprops(glcm, 'contrast')
    dissimilarity = feature.graycoprops(glcm, 'dissimilarity')
    homogeneity = feature.graycoprops(glcm, 'homogeneity')
    energy = feature.graycoprops(glcm, 'energy')
    correlation = feature.graycoprops(glcm, 'correlation')
    asm = feature.graycoprops(glcm, 'ASM')

    glcm = glcm + 1e-10  # Tambahan kecil agar tidak log(0)
    entropy = -np.sum(glcm * np.log2(glcm))

    return np.mean(contrast), np.mean(dissimilarity), np.mean(homogeneity), np.mean(energy), np.mean(correlation), np.mean(asm), entropy

In [4]:
def process_images_in_folder(folder_path, output_csv):
    results = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".png") or filename.endswith(".jpg"):
            image_path = os.path.join(folder_path, filename)
            image = cv2.imread(image_path)

            # Hitung fitur RGB
            mean_R, mean_G, mean_B = extract_rgb_from_image(image)

            # Ekstrak channel warna
            red_channel = img_as_ubyte(image[:, :, 2])
            green_channel = img_as_ubyte(image[:, :, 1])
            blue_channel = img_as_ubyte(image[:, :, 0])

            # Hitung fitur GLCM per channel
            contrast_R, dissimilarity_R, homogeneity_R, energy_R, correlation_R, asm_R, entropy_R = calculate_glcm_features(
                red_channel)
            contrast_G, dissimilarity_G, homogeneity_G, energy_G, correlation_G, asm_G, entropy_G = calculate_glcm_features(
                green_channel)
            contrast_B, dissimilarity_B, homogeneity_B, energy_B, correlation_B, asm_B, entropy_B = calculate_glcm_features(
                blue_channel)

            results.append({
                'filename': filename,
                'mean_R': mean_R,
                'mean_G': mean_G,
                'mean_B': mean_B,
                'contrast_R': contrast_R,
                'dissimilarity_R': dissimilarity_R,
                'homogeneity_R': homogeneity_R,
                'energy_R': energy_R,
                'correlation_R': correlation_R,
                'asm_R': asm_R,
                'entropy_R': entropy_R,
                'contrast_G': contrast_G,
                'dissimilarity_G': dissimilarity_G,
                'homogeneity_G': homogeneity_G,
                'energy_G': energy_G,
                'correlation_G': correlation_G,
                'asm_G': asm_G,
                'entropy_G': entropy_G,
                'contrast_B': contrast_B,
                'dissimilarity_B': dissimilarity_B,
                'homogeneity_B': homogeneity_B,
                'energy_B': energy_B,
                'correlation_B': correlation_B,
                'asm_B': asm_B,
                'entropy_B': entropy_B
            })

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f'Results are saved to {output_csv}')

In [ ]:
def process_all_folders(main_folder_path, output_folder_path):
    os.makedirs(output_folder_path, exist_ok=True)
    for folder_name in os.listdir(main_folder_path):
        folder_path = os.path.join(main_folder_path, folder_name)
        if os.path.isdir(folder_path):
            output_csv = f'{folder_name}_glcm_result.csv'
            output_csv_path = os.path.join(output_folder_path, output_csv)
            process_images_in_folder(folder_path, output_csv_path)


main_folder_path = r'C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\data\runway'
output_folder_path = r'C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\GLCM'
process_all_folders(main_folder_path, output_folder_path)

Results are saved to C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\adage_studio_project_x_unrefyned_glcm_result.csv
Results are saved to C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\adama_paris_glcm_result.csv
Results are saved to C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\ajabeng_glcm_result.csv
Results are saved to C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\ajanee_glcm_result.csv
Results are saved to C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\babayo_glcm_result.csv
Results are saved to C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\bloke_glcm_result.csv
Results are saved to C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\boyedoe_glcm

LBP EXTRACTION

In [6]:
# Parameter LBP
# =========================
LBP_RADIUS = 1
LBP_N_POINTS = 8 * LBP_RADIUS
LBP_METHOD = 'uniform'
LBP_N_BINS = LBP_N_POINTS + 2 if LBP_METHOD == 'uniform' else None  # Konsisten

In [7]:
def calculate_lbp_features(image_channel):
    if image_channel is None:
        return [0]*LBP_N_BINS, 0, 0, 0, 0, 0, 0

    lbp = local_binary_pattern(
        image_channel, LBP_N_POINTS, LBP_RADIUS, LBP_METHOD)

    # Gunakan jumlah bin tetap
    hist, _ = np.histogram(lbp.ravel(), bins=LBP_N_BINS, range=(0, LBP_N_BINS))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-7)

    contrast = np.var(lbp)
    dissimilarity = np.sum(np.abs(np.arange(LBP_N_BINS) - np.mean(lbp)) * hist)
    homogeneity = np.sum(
        hist / (1.0 + np.abs(np.arange(LBP_N_BINS) - np.mean(lbp))))
    energy = np.sum(hist ** 2)

    # Korelasi: tangani error numerik kecil agar hasil tidak negatif tak masuk akal
    raw_correlation = np.sum(
        (np.arange(LBP_N_BINS) - np.mean(lbp)) * hist) / (np.std(lbp) + 1e-7)
    correlation = max(0.0, raw_correlation) if abs(
        raw_correlation) < 1e-6 else raw_correlation

    ent = entropy(hist)

    return hist.tolist(), contrast, dissimilarity, homogeneity, energy, correlation, ent

In [8]:
def process_images_in_folder(folder_path, output_csv_base):
    results_main = []
    results_hist = []

    for filename in tqdm(os.listdir(folder_path), desc=f'Processing {os.path.basename(folder_path)}'):
        if filename.lower().endswith((".png", ".jpg", ".jpeg")):
            image_path = os.path.join(folder_path, filename)
            image = cv2.imread(image_path)

            if image is None or image.shape[0] < 3 or image.shape[1] < 3:
                print(
                    f"Warning: {filename} gagal dibaca atau terlalu kecil. Lewati.")
                continue

            mean_R, mean_G, mean_B = extract_rgb_from_image(image)

            red_channel = image[:, :, 2]
            green_channel = image[:, :, 1]
            blue_channel = image[:, :, 0]

            lbp_R, contrast_R, dissimilarity_R, homogeneity_R, energy_R, correlation_R, entropy_R = calculate_lbp_features(
                red_channel)
            lbp_G, contrast_G, dissimilarity_G, homogeneity_G, energy_G, correlation_G, entropy_G = calculate_lbp_features(
                green_channel)
            lbp_B, contrast_B, dissimilarity_B, homogeneity_B, energy_B, correlation_B, entropy_B = calculate_lbp_features(
                blue_channel)

            result_main = {
                'filename': filename,
                'mean_R': mean_R,
                'mean_G': mean_G,
                'mean_B': mean_B,
                'contrast_R': contrast_R,
                'dissimilarity_R': dissimilarity_R,
                'homogeneity_R': homogeneity_R,
                'energy_R': energy_R,
                'correlation_R': correlation_R,
                'entropy_R': entropy_R,
                'contrast_G': contrast_G,
                'dissimilarity_G': dissimilarity_G,
                'homogeneity_G': homogeneity_G,
                'energy_G': energy_G,
                'correlation_G': correlation_G,
                'entropy_G': entropy_G,
                'contrast_B': contrast_B,
                'dissimilarity_B': dissimilarity_B,
                'homogeneity_B': homogeneity_B,
                'energy_B': energy_B,
                'correlation_B': correlation_B,
                'entropy_B': entropy_B
            }

            results_main.append(result_main)

            # Simpan histogram LBP
            result_hist = {'filename': filename}
            for i in range(LBP_N_BINS):
                result_hist[f'lbp_R_{i}'] = lbp_R[i]
                result_hist[f'lbp_G_{i}'] = lbp_G[i]
                result_hist[f'lbp_B_{i}'] = lbp_B[i]
            results_hist.append(result_hist)

    # Simpan CSV utama
    df_main = pd.DataFrame(results_main)
    df_main.to_csv(f"{output_csv_base}_features.csv", index=False)

    # Simpan CSV histogram
    df_hist = pd.DataFrame(results_hist)
    df_hist.to_csv(f"{output_csv_base}_histogram.csv", index=False)

    print(
        f'Hasil disimpan: {output_csv_base}_features.csv dan {output_csv_base}_histogram.csv')

In [10]:
def process_all_folders(main_folder_path, output_folder_path):
    os.makedirs(output_folder_path, exist_ok=True)
    
    for folder_name in os.listdir(main_folder_path):
        folder_path = os.path.join(main_folder_path, folder_name)
        
        if os.path.isdir(folder_path):
            print(f'Processing folder: {folder_name}')
            
            output_csv_base = os.path.join(output_folder_path, folder_name)
            process_images_in_folder(folder_path, output_csv_base)



main_folder_path = r'C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\data\runway'
output_folder_path = r'C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\LBP'
process_all_folders(main_folder_path, output_folder_path)h


Processing folder: adage_studio_project_x_unrefyned


Processing adage_studio_project_x_unrefyned: 100%|██████████| 5/5 [01:27<00:00, 17.43s/it]


Hasil disimpan: C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\LBP\adage_studio_project_x_unrefyned_features.csv dan C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\LBP\adage_studio_project_x_unrefyned_histogram.csv
Processing folder: adama_paris


Processing adama_paris: 100%|██████████| 15/15 [02:25<00:00,  9.73s/it]


Hasil disimpan: C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\LBP\adama_paris_features.csv dan C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\LBP\adama_paris_histogram.csv
Processing folder: ajabeng


...

Hasil disimpan: C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\LBP\the_or_foundation_features.csv dan C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\LBP\the_or_foundation_histogram.csv
Processing folder: wote


Processing wote: 100%|██████████| 33/33 [01:13<00:00,  2.24s/it]


Hasil disimpan: C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\LBP\wote_features.csv dan C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\LBP\wote_histogram.csv
Processing folder: y'wande


Processing y'wande: 100%|██████████| 34/34 [01:15<00:00,  2.23s/it]

Hasil disimpan: C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\LBP\y'wande_features.csv dan C:\Users\User\OneDrive\Desktop\Lagos-FW-2024-Analysis\Lagos-FW-2024-Analysis-1\outputs\features\LBP\y'wande_histogram.csv
